# Table detection example

In [ ]:
arsenal_config_path = "@comp/zemi/llm_curated_set_model_mode.toml"
arsenal_start_and_stop_at_job_level = False
model_name = "qwen35_4b"
temperature = 0.0
max_tokens = 1024
encoding_prompt = {'prompt_name': 'cells_basic', 'prompt_file': '@comp/optimizer_example/params/prompts.md', 'encoder': '@comp/optimizer_example/params/encoder.py:encode', 'encoding_format': 'cells'}
dataset_input = {"workbook_path": "@comp/data/validation/single_table.xlsx", "worksheet_name": "Sheet1"}

In [ ]:
from pathlib import Path
import sys

component_root = Path.cwd()
while not (component_root / ".zemicomp").is_file():
    if component_root.parent == component_root:
        raise FileNotFoundError("Could not find the ZEMI component root")
    component_root = component_root.parent
sys.path.insert(0, str(component_root))
from zemi.prompting import build_prompt
item_text, prompt = build_prompt(encoding_prompt, dataset_input["workbook_path"], dataset_input["worksheet_name"])


In [ ]:
from zemi.arsenal import ArsenalSession
import zemi

session = ArsenalSession(arsenal_config_path)
if not arsenal_start_and_stop_at_job_level:
    zemi.arsenal.begin(session, stop_before_begin=True)
assistant = session.model(model_name).assistants["assistant"]
client = assistant.clients.openai.client


In [ ]:
import json
import time
from urllib.parse import urlsplit, urlunsplit
from urllib.request import Request, urlopen
from zemi.playbook import output_params

base_url = urlsplit(str(client.base_url))
tokenize_url = urlunsplit((base_url.scheme, base_url.netloc, "/tokenize", "", ""))
def token_count(text):
    request = Request(tokenize_url, data=json.dumps({"content": text, "add_special": False}).encode("utf-8"), headers={"Content-Type": "application/json"})
    with urlopen(request, timeout=60) as reply:
        return len(json.load(reply)["tokens"])
item_tokens = token_count(item_text)
started = time.perf_counter()
try:
    response = client.chat.completions.create(model=assistant.clients.model,
        messages=[{"role": "user", "content": prompt}], temperature=temperature, max_tokens=max_tokens)
    lm_time = round(time.perf_counter() - started, 3)
    raw = response.choices[0].message.content or ""
    outputs = {"raw_response": raw, "lm_time": lm_time, "item_tokens": item_tokens,
               "prompt_tokens": response.usage.prompt_tokens if response.usage else None}
    try:
        result = json.loads(raw)
        if (not isinstance(result, dict) or set(result) != {"ranges"}
                or not isinstance(result["ranges"], list)
                or not all(isinstance(value, str) for value in result["ranges"])):
            raise ValueError('Expected a JSON object with a string ranges array')
    except (ValueError, TypeError):
        output_params(outputs, report=["lm_time", "item_tokens", "prompt_tokens"])
        raise
    outputs["ranges"] = result["ranges"]
    output_params(outputs, report=["ranges", "lm_time", "item_tokens", "prompt_tokens"])
finally:
    if not arsenal_start_and_stop_at_job_level:
        zemi.arsenal.end(session, stop_after_end=True)
